# Ноутбук с финальным моделированием

## 1. Импорт бибилиотек и конфигурация проекта

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import Ridge
from category_encoders.cat_boost import CatBoostEncoder
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, KFold, RandomizedSearchCV
import copy
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_validate
import xgboost as xgb
import pyarrow
import phik
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import catboost
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate
)
from catboost import CatBoostRegressor
import mlflow.sklearn
from phik.report import plot_correlation_matrix
import plotly
import mlflow
import time
from scipy.stats import randint, uniform, loguniform
import os
from datetime import datetime
import category_encoders as ce
import joblib

c:\Users\Степан\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Создадим словарь конфигураций.

CONFIG = {
    # Константы
    "DEV_MODE": False,
    "DEV_SAMPLE_SIZE": 100000,
    "RANDOM_STATE": 42,
    # Целевая переменная 
    "TARGET": "Цена",
    "YEAR": datetime.now().year
}

In [3]:
df = pd.read_parquet("../data/raw/df_optimal.parquet")

## 2. Очистка данных

In [4]:
df = df.drop_duplicates()
df['Есть особые отметки'] = df['Особые отметки'].notna().astype(int)
df = df[df['Год'] >= 1990]
df = df[(df['Объем двигателя'] == 0) | ((df['Объем двигателя'] >= 0.6) & (df['Объем двигателя'] <= 6.8))]
df = df[df['Мощность'] <= 800]
df = df[df['Руль'] != 'правый, левый']
df = df[df['Цена'] >= 150000]

In [5]:
owners_mapping = {
    '4 и более': 4, # Стандартное упрощение для моделей. Оно дает ей понять направление тренда (что владельцев много), не усложняя вычисления.
    '1.0': 1,
    '2.0': 2,
    '3.0': 3,
    1.0: 1,
    2.0: 2,
    3.0: 3,
    '1': 1,
    '2': 2,
    '3': 3
}

# Применяем замену к столбцу
df['Владельцы'] = df['Владельцы'].replace(owners_mapping)

C:\Users\Степан\AppData\Local\Temp\ipykernel_20164\1925271674.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Владельцы'] = df['Владельцы'].replace(owners_mapping)


In [6]:
threshold = len(df) * 0.1
df = df.dropna(thresh=threshold, axis=1).copy()

In [7]:
df = df.drop(columns=['Макро-регион', 'Город', 'Пропуски в данных', 'Ссылка', 'Кол-во просмотров', 'Ошибка_ст', 'Ошибка_знач', 'Скрыто'])

In [8]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

In [9]:
df.isna().sum()

Название машины                    0
Год                                0
Дата размещения объявления         0
Цена                               0
Объем двигателя                    0
Тип двигателя                      0
Мощность                           0
Коробка передач                    0
Привод                             0
Пробег                          3942
Руль                               0
Поколение                         55
Рестайлинг                        55
Цвет                               0
Комплектация                       0
Владелец                           0
Тип кузова                         0
Метка                              0
Регион                             0
Владельцы                     223481
Есть особые отметки                0
dtype: int64

## 3. Генерация признаков

In [10]:
def get_brand_tier(brand):
    luxury = {
        "rolls-royce",
        "bentley",
        "lamborghini",
        "ferrari",
        "aston_martin",
        "aston martin",
        "maserati",
        "bugatti",
        "mclaren",
    }

    premium = {
        "bmw",
        "mercedes-benz",
        "audi",
        "lexus",
        "porsche",
        "land_rover",
        "land rover",
        "jaguar",
        "genesis",
        "infiniti",
        "cadillac",
        "volvo",
        "jeep",
        "hongqi",
        "li",
    }

    economy = {"lada", "daewoo", "zaz", "gaz", "uaz", "vaz"}

    if brand in luxury:
        return "Люкс"
    elif brand in premium:
        return "Премиум"
    elif brand in economy:
        return "Эконом"
    else:
        # Сюда попадут все остальные: toyota, nissan, hyundai, kia, volkswagen,
        # chery, geely, haval, peugeot, citroen, subaru, skoda и т.д.
        return "Масс-маркет"


In [11]:
def create_features(df):
    
    df = df.copy()
    df = df.rename(columns={'Метка': 'Марка'})

    df['Литровая мощность'] = (df['Мощность'] / df['Объем двигателя'].replace(0, np.nan)).fillna(0)  # Чтобы избежать деления на ноль
    # Уберем "Название машины", заменив на "Модель"
    df['Название машины'] = df['Название машины'].astype(str)
    df['Марка'] = df['Марка'].astype(str)
    def extract_model(row):
        full_name = row['Название машины']
        # Нормализуем исходную марку (например, "aston_martin" -> "aston martin")
        brand_raw = row['Марка'].lower().replace('_', ' ')
        full_name_lower = full_name.lower()
        
        # 1. Словарь синонимов для брендов, которые могут быть написаны по-русски
        brand_synonyms = {
            'lada': ['лада'],
            'uaz': ['уаз'],
            'gaz': ['газ'],
            'moskvich': ['москвич'],
        }
        
        # 2. Получаем список возможных написаний для текущей марки.
        # Если марки нет в словаре синонимов, используем только саму марку (в списке).
        possible_names = brand_synonyms.get(brand_raw, [brand_raw])
        
        # 3. Проверяем каждый синоним по очереди
        for name in possible_names:
            if full_name_lower.startswith(name):
                # Как только нашли совпадение, отрезаем его длину от оригинального названия
                return full_name[len(name):].strip()
                
        # Если совпадений вообще не нашлось (например, написано просто "Гранта")
        return full_name

    # Применяем новую функцию
    df['Модель'] = df.apply(extract_model, axis=1)

    cat_cols = df.select_dtypes('object').columns.to_list()
    df[cat_cols] = df[cat_cols].astype('category')


    if 'Владелец' in df.columns:
        def clean_seller_type(text):
            text = str(text).lower()
            if 'фирма' in text or 'дилер' in text or 'компания' in text:
                return 'Дилер/Салон'
            elif 'частное лицо' in text or 'частник' in text:
                return 'Частное лицо'
            else:
                return 'Unknown'
                
    df['Тип продавца'] = df['Владелец'].apply(clean_seller_type)
    df = df.drop(columns=['Владелец']) # удаляем исходный грязный столбец
    df['Класс бренда'] = df['Марка'].apply(get_brand_tier)

    if 'Дата размещения объявления' in df.columns:
        df['Дата размещения объявления'] = pd.to_datetime(df['Дата размещения объявления'], format='mixed', errors='coerce')
        df['Год размещения'] = df['Дата размещения объявления'].dt.year
        df = df.drop(columns=['Дата размещения объявления'])

    df['Возраст'] = df['Год размещения'] - df['Год']
    df['Пробег за год'] = df['Пробег'] / df['Возраст'].replace(0, 1)  # Чтобы избежать деления на ноль

    df['Рестайлинг'] = df['Рестайлинг'].fillna('Unknown').astype(str).astype('category')
    df['Поколение'] = df['Поколение'].fillna('Unknown').astype(str).astype('category')

    return df

In [12]:
df = create_features(df)

In [13]:
df.drop(columns=['Возраст', 'Название машины'], inplace=True)

## 3. Обучение модели

In [14]:
cat_cols = df.select_dtypes(include=['category', 'string', 'object']).columns.tolist()

In [15]:
text_features = ['Комплектация']
for col in text_features:
    df[col] = df[col].astype(str)

In [16]:
X, y = df.drop(columns=[CONFIG["TARGET"]]), df[CONFIG["TARGET"]]

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=CONFIG["RANDOM_STATE"],
    shuffle=True
)

In [19]:
cv = KFold(n_splits=5, shuffle=True, random_state=CONFIG["RANDOM_STATE"])
params = {
    'bootstrap_type': 'Bernoulli',
    'iterations': 2500,
    'learning_rate': 0.042248138023520114,
    'depth': 10,
    'l2_leaf_reg': 9.394802632887766,
    'random_strength': 0.8303775333704183,
    'border_count': 207,
    'subsample': 0.5483853226992782,
    'loss_function': 'MAE',
    'allow_writing_files': False,
    'eval_metric': 'MAE',
    'random_seed': 42,
    'thread_count': -1,
    'task_type': 'GPU',
    'verbose': 0
}
scoring = {
    'mape': 'neg_mean_absolute_percentage_error',
    'mae': 'neg_mean_absolute_error'
}

In [20]:
model = CatBoostRegressor(**params)
wrapped_model = TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )

In [21]:
scores = cross_validate(
            wrapped_model,
            X_train,
            y_train,
            cv=cv,
            params={'cat_features': cat_cols, 'text_features': text_features},
            scoring=scoring
        )

cv_mape = -scores['test_mape'].mean()
cv_mae = -scores['test_mae'].mean()
wrapped_model.fit(X_train, y_train, cat_features=cat_cols, text_features=text_features)
y_pred_test = wrapped_model.predict(X_test)

test_mape = mean_absolute_percentage_error(y_test, y_pred_test)
test_mae = mean_absolute_error(y_test, y_pred_test)
cheap_mask = y_test < 300000
print("MAPE на дешевых (<300к):", mean_absolute_percentage_error(y_test[cheap_mask], y_pred_test[cheap_mask]))
print("MAPE на остальных (>300к):", mean_absolute_percentage_error(y_test[~cheap_mask], y_pred_test[~cheap_mask]))
print(f"\nCV MAPE: {cv_mape:.4f} | CV MAE: {cv_mae:.0f} руб.\nTEST MAPE: {test_mape:.4f} | TEST MAE: {test_mae:.0f} руб.\n")

Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


MAPE на дешевых (<300к): 0.23706720789483773
MAPE на остальных (>300к): 0.13772924695601257

CV MAPE: 0.1557 | CV MAE: 147643 руб.
TEST MAPE: 0.1539 | TEST MAE: 142810 руб.



In [22]:
print(type(wrapped_model))

<class 'sklearn.compose._target.TransformedTargetRegressor'>


In [23]:
joblib.dump(wrapped_model, '../models/best_catboost_model.pkl')

['../models/best_catboost_model.pkl']

In [24]:
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_parquet("../data/optimized/train_optimized.parquet", index=False)
test_df.to_parquet("../data/optimized/test_optimized.parquet", index=False)